In [2]:
import os, json
import pandas as pd


In [38]:
homes = pd.read_csv('processed/NY_SLI_ATTOM_NEW.csv')
homes_raw = []
zips = []
with open('raw/home_jsons.json') as f:
    homes_raw = json.load(f)
with open('raw/zip_data.json') as f:
    zips = json.load(f)


In [59]:
homes_raw.sort(key=lambda x: x['status']['attomId'])
zips.sort(key=lambda x: x['community']['geography']['geoIdV4'])

def binary_search_homes(homes, target_attom_id):
    """
    homes: list of dicts sorted by homes[i]["identifier"]["attomId"]
    target_attom_id: int
    Returns the index if found, else -1
    """
    low, high = 0, len(homes) - 1

    while low <= high:
        mid = (low + high) // 2
        mid_id = homes[mid]["status"]["attomId"]

        if mid_id == target_attom_id:
            return mid
        elif mid_id < target_attom_id:
            low = mid + 1
        else:
            high = mid - 1

    return -1
def binary_search_zips(zips, target_attom_id):
    """
    homes: list of dicts sorted by homes[i]["identifier"]["attomId"]
    target_attom_id: int
    Returns the index if found, else -1
    """
    low, high = 0, len(zips) - 1

    while low <= high:
        mid = (low + high) // 2
        mid_id = zips[mid]['community']['geography']['geoIdV4']

        if mid_id == target_attom_id:
            return mid
        elif mid_id < target_attom_id:
            low = mid + 1
        else:
            high = mid - 1

    return -1


for idx, row in homes.iterrows():
    attomID = row['attomID']
    idx = binary_search_homes(homes_raw, attomID)
    geoids = homes_raw[idx]['property'][0]['location']['geoIdV4']
    zip = 0
    try:
        zip = geoids['ZI']
    except KeyError as e:
        print('bruh')
        continue
    community_idx = binary_search_zips(zips, zip)
    
    stats = zips[community_idx]['community']
    for tp, vl in stats.items():
        if tp == 'geography':
            continue
        for colname, data in vl.items():
            homes.at[idx, colname] = data
    

bruh
bruh


In [63]:
homes.to_csv('processed/NY_SLI_ATTOM_EXTENDED.csv')